# PathSnap GPS Map-Matching — Exploratory Data Analysis

This notebook covers:
1. Data loading utilities for all four file types
2. Dataset statistics (trip count, GPS obs, nodes, arcs)
3. Missing value & timestamp gap analysis
4. GPS trajectory visualisation (matplotlib)
5. Road-network connectivity check
6. Interactive Folium map (raw GPS + ground-truth route)
7. Anomaly detection (large gaps, out-of-range coordinates)
8. Distribution plots for key statistics

## 0 · Setup & imports

In [ ]:
import os
import sys
import glob
import math
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import matplotlib.colors as mcolors
from scipy import stats

try:
    import folium
    HAS_FOLIUM = True
except ImportError:
    HAS_FOLIUM = False
    warnings.warn('folium not installed – interactive map cell will be skipped.')

warnings.filterwarnings('ignore')
plt.rcParams.update({'figure.dpi': 110, 'font.size': 11})

# ── Adjust this to point at your data root ────────────────────────────────
REPO_ROOT   = Path(os.getcwd()).resolve().parents[0]   # pathsnap/
DATA_ROOT   = REPO_ROOT.parent / 'data'               # adjust if needed
TRAIN_DIR   = DATA_ROOT / 'train'
STEST_DIR   = DATA_ROOT / 'student_test'
EVAL_DIR    = DATA_ROOT / 'evaluator_test'             # leave untouched

print('Repo root :', REPO_ROOT)
print('Data root :', DATA_ROOT)
print('Train dir :', TRAIN_DIR)
print('Student test dir:', STEST_DIR)

## 1 · File-loading utilities

We import the production loaders from `src/preprocessing/data_loader.py`.  
Inline implementations are shown below for reference, but only the import is used.

In [ ]:
# Add src to path so we can import our modules
src_path = str(REPO_ROOT / 'src')
if src_path not in sys.path:
    sys.path.insert(0, src_path)

from preprocessing.data_loader import (
    load_track, load_nodes, load_arcs, load_route,
    load_trip, load_partition,
)

print('Loaders imported successfully.')

In [ ]:
# ── Inline reference implementations (documentation only) ─────────────────

def _ref_load_track(path):
    """lon lat timestamp — whitespace-separated, no header."""
    return pd.read_csv(path, sep=r'\s+', header=None,
                       names=['lon', 'lat', 'timestamp'],
                       dtype={'lon': float, 'lat': float, 'timestamp': float})

def _ref_load_nodes(path):
    """lon lat — row index = node_id."""
    df = pd.read_csv(path, sep=r'\s+', header=None,
                     names=['lon', 'lat'], dtype=float)
    df.index.name = 'node_id'
    return df

def _ref_load_arcs(path):
    """from_node_id to_node_id — row index = arc_id."""
    df = pd.read_csv(path, sep=r'\s+', header=None,
                     names=['from_node_id', 'to_node_id'], dtype=int)
    df.index.name = 'arc_id'
    return df

def _ref_load_route(path):
    """One arc_id per line."""
    with open(path) as f:
        return [int(l.strip()) for l in f if l.strip()]

print('Reference loaders defined (not used in EDA).')

## 2 · Load all training trips

In [ ]:
if not TRAIN_DIR.exists():
    print(f'[WARNING] Training directory not found: {TRAIN_DIR}')
    print('Creating synthetic data for demonstration …')

    import tempfile, textwrap, shutil
    _tmp_root = Path(tempfile.mkdtemp())
    TRAIN_DIR = _tmp_root / 'train'

    rng = np.random.default_rng(42)
    for trip_i in range(5):
        td = TRAIN_DIR / f'trip_{trip_i:03d}'
        td.mkdir(parents=True)
        n_obs = rng.integers(20, 60)
        base_lon, base_lat = -122.4 + rng.uniform(-0.05, 0.05), 37.77 + rng.uniform(-0.05, 0.05)
        lons = base_lon + np.cumsum(rng.normal(0, 0.0003, n_obs))
        lats = base_lat + np.cumsum(rng.normal(0, 0.0002, n_obs))
        ts   = 1609459200 + np.arange(n_obs)
        track_text = '\n'.join(f'{lo:.6f} {la:.6f} {t}' for lo, la, t in zip(lons, lats, ts))
        (td / f'trip_{trip_i:03d}.track').write_text(track_text)

        n_nodes = 10
        node_lons = base_lon + rng.uniform(-0.01, 0.01, n_nodes)
        node_lats = base_lat + rng.uniform(-0.01, 0.01, n_nodes)
        nodes_text = '\n'.join(f'{lo:.6f} {la:.6f}' for lo, la in zip(node_lons, node_lats))
        (td / f'trip_{trip_i:03d}.nodes').write_text(nodes_text)

        arcs = [(j, j+1) for j in range(n_nodes - 1)]
        arcs_text = '\n'.join(f'{a} {b}' for a, b in arcs)
        (td / f'trip_{trip_i:03d}.arcs').write_text(arcs_text)

        route_ids = rng.choice(len(arcs), size=min(n_obs, len(arcs)), replace=True)
        (td / f'trip_{trip_i:03d}.route').write_text('\n'.join(str(x) for x in route_ids))

    print(f'Synthetic data written to {TRAIN_DIR}')
else:
    _tmp_root = None

trips = load_partition(TRAIN_DIR)
print(f'Loaded {len(trips)} training trips.')

## 3 · Dataset statistics

In [ ]:
stats_rows = []
for t in trips:
    track = t['track']
    nodes = t['nodes']
    arcs  = t['arcs']
    route = t['route']

    dt = track['timestamp'].diff().dropna()
    stats_rows.append({
        'trip_id'       : t['trip_id'],
        'gps_obs'       : len(track),
        'n_nodes'       : len(nodes),
        'n_arcs'        : len(arcs),
        'route_len'     : len(route) if route is not None else None,
        'ts_min'        : track['timestamp'].min(),
        'ts_max'        : track['timestamp'].max(),
        'duration_s'    : track['timestamp'].max() - track['timestamp'].min(),
        'mean_dt_s'     : dt.mean(),
        'max_dt_s'      : dt.max(),
        'lon_min'       : track['lon'].min(),
        'lon_max'       : track['lon'].max(),
        'lat_min'       : track['lat'].min(),
        'lat_max'       : track['lat'].max(),
        'nan_count'     : track[['lon','lat','timestamp']].isna().sum().sum(),
    })

stats_df = pd.DataFrame(stats_rows)
print('=== Per-trip statistics ===')
display(stats_df.describe())

In [ ]:
print(f'Total trips          : {len(trips)}')
print(f'GPS obs — median     : {stats_df["gps_obs"].median():.0f}')
print(f'GPS obs — range      : {stats_df["gps_obs"].min()} – {stats_df["gps_obs"].max()}')
print(f'Nodes   — median     : {stats_df["n_nodes"].median():.0f}')
print(f'Arcs    — median     : {stats_df["n_arcs"].median():.0f}')
print(f'Trips with route     : {stats_df["route_len"].notna().sum()}')
print(f'Total NaN values     : {stats_df["nan_count"].sum()}')

## 4 · Missing value & timestamp gap analysis

In [ ]:
all_dts = []
large_gap_trips = []
LARGE_GAP_THRESHOLD = 30  # seconds

for t in trips:
    track = t['track']
    dts = track['timestamp'].diff().dropna().values
    all_dts.extend(dts.tolist())
    n_large = (dts > LARGE_GAP_THRESHOLD).sum()
    if n_large > 0:
        large_gap_trips.append({'trip_id': t['trip_id'], 'n_large_gaps': n_large,
                                 'max_gap_s': dts.max()})

print(f'Trips with gaps > {LARGE_GAP_THRESHOLD}s: {len(large_gap_trips)}')
if large_gap_trips:
    display(pd.DataFrame(large_gap_trips))

all_dts = np.array(all_dts)
print(f'\nTimestamp gap stats (all trips):')
print(f'  mean={all_dts.mean():.2f}s  median={np.median(all_dts):.2f}s  '
      f'max={all_dts.max():.2f}s  std={all_dts.std():.2f}s')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Left: histogram of timestamp gaps
ax = axes[0]
clipped = all_dts[all_dts <= 60]
ax.hist(clipped, bins=60, color='steelblue', edgecolor='white', linewidth=0.4)
ax.axvline(LARGE_GAP_THRESHOLD, color='crimson', lw=1.5, ls='--',
           label=f'Large-gap threshold ({LARGE_GAP_THRESHOLD}s)')
ax.set_xlabel('Timestamp gap (s)')
ax.set_ylabel('Count')
ax.set_title('Distribution of GPS timestamp gaps (clipped at 60s)')
ax.legend()

# Right: missing values per trip
ax = axes[1]
ax.bar(range(len(stats_df)), stats_df['nan_count'], color='salmon')
ax.set_xlabel('Trip index')
ax.set_ylabel('NaN count')
ax.set_title('NaN values per trip (lon + lat + timestamp)')

plt.tight_layout()
plt.show()

## 5 · GPS trajectory plots

In [ ]:
n_show = min(9, len(trips))
ncols  = 3
nrows  = math.ceil(n_show / ncols)

fig, axes = plt.subplots(nrows, ncols, figsize=(5 * ncols, 4 * nrows))
axes = axes.flatten() if n_show > 1 else [axes]

for i, trip in enumerate(trips[:n_show]):
    ax = axes[i]
    track = trip['track']
    # Colour by time progression
    norm  = mcolors.Normalize(vmin=track['timestamp'].min(), vmax=track['timestamp'].max())
    cmap  = cm.get_cmap('viridis')
    colors = cmap(norm(track['timestamp'].values))
    ax.scatter(track['lon'], track['lat'], c=colors, s=8, linewidths=0)
    ax.plot(track['lon'], track['lat'], color='grey', lw=0.4, alpha=0.5)
    ax.set_title(f"{trip['trip_id']}  (n={len(track)})", fontsize=9)
    ax.set_xlabel('lon', fontsize=8)
    ax.set_ylabel('lat', fontsize=8)
    ax.tick_params(labelsize=7)

for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

fig.suptitle('GPS trajectories (colour = time progression)', y=1.01, fontsize=13)
plt.tight_layout()
plt.show()

## 6 · Road-network connectivity check

In [ ]:
def check_connectivity(nodes_df: pd.DataFrame, arcs_df: pd.DataFrame) -> dict:
    """Return connectivity statistics for a road network.

    Uses union-find to identify connected components in the undirected
    version of the arc graph.

    Returns
    -------
    dict with keys: n_nodes, n_arcs, n_components, isolated_nodes,
                    largest_component_size
    """
    n = len(nodes_df)
    parent = list(range(n))

    def find(x):
        while parent[x] != x:
            parent[x] = parent[parent[x]]
            x = parent[x]
        return x

    def union(x, y):
        rx, ry = find(x), find(y)
        if rx != ry:
            parent[ry] = rx

    for _, row in arcs_df.iterrows():
        u, v = int(row['from_node_id']), int(row['to_node_id'])
        if 0 <= u < n and 0 <= v < n:
            union(u, v)

    roots = [find(i) for i in range(n)]
    from collections import Counter
    comp_sizes = Counter(roots)
    isolated = sum(1 for sz in comp_sizes.values() if sz == 1)

    return {
        'n_nodes'               : n,
        'n_arcs'                : len(arcs_df),
        'n_components'          : len(comp_sizes),
        'isolated_nodes'        : isolated,
        'largest_component_size': max(comp_sizes.values()),
    }

conn_rows = []
for t in trips:
    info = check_connectivity(t['nodes'], t['arcs'])
    info['trip_id'] = t['trip_id']
    conn_rows.append(info)

conn_df = pd.DataFrame(conn_rows)
display(conn_df[['trip_id','n_nodes','n_arcs','n_components',
                  'isolated_nodes','largest_component_size']])
print(f'\nTrips with isolated nodes: {(conn_df["isolated_nodes"] > 0).sum()}')
print(f'Trips with >1 component  : {(conn_df["n_components"] > 1).sum()}')

## 7 · Folium interactive map — one trip

In [ ]:
TRIP_IDX = 0  # change this to inspect a different trip

if not HAS_FOLIUM:
    print('folium not installed — skipping interactive map.')
else:
    t      = trips[TRIP_IDX]
    track  = t['track']
    nodes  = t['nodes']
    arcs   = t['arcs']
    route  = t['route']

    center_lat = track['lat'].mean()
    center_lon = track['lon'].mean()

    m = folium.Map(location=[center_lat, center_lon], zoom_start=15,
                   tiles='OpenStreetMap')

    # Raw GPS points
    gps_group = folium.FeatureGroup(name='GPS points')
    for _, row in track.iterrows():
        folium.CircleMarker(
            location=[row['lat'], row['lon']],
            radius=3,
            color='dodgerblue',
            fill=True,
            fill_opacity=0.7,
            tooltip=f"ts={row['timestamp']:.0f}",
        ).add_to(gps_group)
    gps_group.add_to(m)

    # GPS trajectory polyline
    track_coords = [[r['lat'], r['lon']] for _, r in track.iterrows()]
    folium.PolyLine(track_coords, color='dodgerblue', weight=1.5,
                    opacity=0.6, name='GPS track').add_to(m)

    # Ground-truth route arcs
    if route is not None:
        route_group = folium.FeatureGroup(name='Ground-truth route')
        for arc_id in route:
            if arc_id not in arcs.index:
                continue
            arc_row = arcs.loc[arc_id]
            try:
                n_from = nodes.loc[arc_row['from_node_id']]
                n_to   = nodes.loc[arc_row['to_node_id']]
            except KeyError:
                continue
            folium.PolyLine(
                [[n_from['lat'], n_from['lon']], [n_to['lat'], n_to['lon']]],
                color='tomato', weight=4, opacity=0.8,
                tooltip=f'arc {arc_id}'
            ).add_to(route_group)
        route_group.add_to(m)

    # All road-network arcs (lighter)
    all_arcs_group = folium.FeatureGroup(name='All arcs', show=False)
    for arc_id, arc_row in arcs.iterrows():
        try:
            n_from = nodes.loc[arc_row['from_node_id']]
            n_to   = nodes.loc[arc_row['to_node_id']]
        except KeyError:
            continue
        folium.PolyLine(
            [[n_from['lat'], n_from['lon']], [n_to['lat'], n_to['lon']]],
            color='grey', weight=1, opacity=0.4
        ).add_to(all_arcs_group)
    all_arcs_group.add_to(m)

    folium.LayerControl().add_to(m)

    display(m)

## 8 · Anomaly detection

In [ ]:
EARTH_R = 6_371_000.0

def haversine_arr(lat1, lon1, lat2, lon2):
    """Vectorised haversine distance (NumPy arrays, degrees in → metres out)."""
    phi1, phi2 = np.radians(lat1), np.radians(lat2)
    dphi = np.radians(lat2 - lat1)
    dlam = np.radians(lon2 - lon1)
    a = np.sin(dphi/2)**2 + np.cos(phi1)*np.cos(phi2)*np.sin(dlam/2)**2
    return 2 * EARTH_R * np.arcsin(np.sqrt(np.clip(a, 0, 1)))

anomaly_rows = []
SPEED_THRESHOLD = 55.0   # m/s (~200 km/h)
COORD_RANGE = dict(lon=(-180, 180), lat=(-90, 90))

for t in trips:
    track = t['track']
    tid   = t['trip_id']

    # Out-of-range coordinates
    bad_lon = ((track['lon'] < -180) | (track['lon'] > 180)).sum()
    bad_lat = ((track['lat'] < -90)  | (track['lat'] > 90)).sum()

    # Speed anomalies
    if len(track) > 1:
        dists = haversine_arr(
            track['lat'].values[:-1], track['lon'].values[:-1],
            track['lat'].values[1:],  track['lon'].values[1:],
        )
        dts   = np.diff(track['timestamp'].values)
        speeds = np.where(dts > 0, dists / dts, 0)
        fast_obs = (speeds > SPEED_THRESHOLD).sum()
        max_speed = speeds.max()
    else:
        fast_obs = 0
        max_speed = 0.0

    anomaly_rows.append({
        'trip_id'     : tid,
        'bad_lon'     : bad_lon,
        'bad_lat'     : bad_lat,
        'fast_obs'    : fast_obs,
        'max_speed_ms': max_speed,
    })

anomaly_df = pd.DataFrame(anomaly_rows)
print('=== Anomaly summary ===')
display(anomaly_df)
print(f'\nTrips with out-of-range lon : {(anomaly_df["bad_lon"] > 0).sum()}')
print(f'Trips with out-of-range lat : {(anomaly_df["bad_lat"] > 0).sum()}')
print(f'Trips with speed > {SPEED_THRESHOLD} m/s: {(anomaly_df["fast_obs"] > 0).sum()}')

## 9 · Distribution plots

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 9))

# 1. GPS obs per trip
ax = axes[0, 0]
ax.hist(stats_df['gps_obs'], bins=20, color='steelblue', edgecolor='white')
ax.set_title('GPS observations per trip')
ax.set_xlabel('GPS obs count')
ax.set_ylabel('Trip count')

# 2. Trip duration
ax = axes[0, 1]
ax.hist(stats_df['duration_s'] / 60, bins=20, color='mediumseagreen', edgecolor='white')
ax.set_title('Trip duration (minutes)')
ax.set_xlabel('Duration (min)')
ax.set_ylabel('Trip count')

# 3. Nodes per trip
ax = axes[0, 2]
ax.hist(stats_df['n_nodes'], bins=20, color='mediumpurple', edgecolor='white')
ax.set_title('Nodes per trip')
ax.set_xlabel('Node count')
ax.set_ylabel('Trip count')

# 4. Arcs per trip
ax = axes[1, 0]
ax.hist(stats_df['n_arcs'], bins=20, color='darkorange', edgecolor='white')
ax.set_title('Arcs per trip')
ax.set_xlabel('Arc count')
ax.set_ylabel('Trip count')

# 5. All GPS timestamp gaps
ax = axes[1, 1]
clipped_dts = all_dts[(all_dts > 0) & (all_dts <= 30)]
ax.hist(clipped_dts, bins=30, color='salmon', edgecolor='white')
ax.set_title('GPS timestamp gaps (≤30 s)')
ax.set_xlabel('Gap (s)')
ax.set_ylabel('Count')

# 6. Max speed per trip
ax = axes[1, 2]
ax.hist(anomaly_df['max_speed_ms'], bins=20, color='teal', edgecolor='white')
ax.axvline(SPEED_THRESHOLD, color='crimson', lw=1.5, ls='--',
           label=f'Anomaly threshold ({SPEED_THRESHOLD} m/s)')
ax.set_title('Max implied speed per trip')
ax.set_xlabel('Speed (m/s)')
ax.set_ylabel('Trip count')
ax.legend(fontsize=8)

plt.suptitle('PathSnap Dataset — Key distributions', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

## 10 · Arc-length distribution

In [ ]:
arc_lengths = []
for t in trips:
    nodes_df = t['nodes']
    for _, arc in t['arcs'].iterrows():
        try:
            na = nodes_df.loc[arc['from_node_id']]
            nb = nodes_df.loc[arc['to_node_id']]
            d = haversine_arr(
                np.array([na['lat']]), np.array([na['lon']]),
                np.array([nb['lat']]), np.array([nb['lon']])
            )[0]
            arc_lengths.append(d)
        except KeyError:
            pass

arc_lengths = np.array(arc_lengths)
print(f'Arc length stats (metres): mean={arc_lengths.mean():.1f}  '
      f'median={np.median(arc_lengths):.1f}  '
      f'max={arc_lengths.max():.1f}')

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(arc_lengths[arc_lengths < np.percentile(arc_lengths, 99)],
        bins=50, color='cornflowerblue', edgecolor='white')
ax.set_title('Arc length distribution (99th-percentile clipped)')
ax.set_xlabel('Arc length (m)')
ax.set_ylabel('Count')
plt.tight_layout()
plt.show()

## 11 · Summary & next steps

| Item | Value |
|------|-------|
| Training trips | see §3 |
| Median GPS obs / trip | see §3 |
| NaN issues | see §3 & §4 |
| Large timestamp gaps (>30 s) | see §4 |
| Isolated / disconnected nodes | see §6 |
| Speed anomalies | see §8 |

**Next steps:**
1. Run `GPSPreprocessor` to drop invalid points and smooth noisy fixes.
2. Call `engineer_features(trip, k=5)` to build the (X, y) matrix for each trip.
3. Concatenate matrices and train a classifier (Random Forest / GBT / LightGBM).
4. At inference time, pick the highest-scoring candidate arc per GPS observation and stitch into a connected route via Viterbi or greedy beam search.

See `src/features/feature_engineering.py` for the full feature schema (13 columns).